In the name of God

Seyed Mani Mirshabani
810801080

# **CA 2, LLMs Spring 2026**


## 1) Prompt Engineering, In-Context Learning, and Instruction Tuning (40 points)

### Initial Setup 

If there is any ambiguity, you may refer to the official documentation on the website (https://devneeds.ir).

In [1]:
import os

os.environ["HF_ENDPOINT"] = "https://hf.devneeds.ir"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "300"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
os.environ["HF_HUB_DISABLE_XET"] = "1"

In [2]:
## Your answer here

### Load Model (2 points)

##### You MUST use the "google/gemma-3-270m-it" model for every task in this section.

In [2]:
from huggingface_hub import snapshot_download

## Your answer here
snapshot_download(repo_id="google/gemma-3-270m-it")

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

'C:\\Users\\SMani24\\.cache\\huggingface\\hub\\models--google--gemma-3-270m-it\\snapshots\\ac82b4e820549b854eebf28ce6dedaf9fdfa17b3'

In [3]:
from transformers import pipeline

## Your answer here

pipe = pipeline(task="text-generation", model="google/gemma-3-270m-it")

Device set to use cuda:0


---

### Prompt Engineering (8 points)

Modern chat-based large language models (LLMs), use a structured message interface rather than a single unstructured prompt string. In this interface, inputs are represented as an ordered list of messages (It is also called ```ChatML format```), where each message is a dictionary (or object) with at least two fields:<br>

role: specifies the source of the message (e.g., system, user, assistant)<br>
content: contains the natural language text of the message<br>

Now search about using this format and define a function named ```ask_llm``` to make it easy to talk to the model. The function must creates messages with two roles: system (general instructions) and user (the question). It sends these to the model and returns the answer. This helps us keep the same format and reuse the function in next tasks.

In [4]:
## Your answer here
def ask_llm(system_instruction, user_question):
    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": user_question}
    ]
    
    output = pipe(
        messages, 
        max_new_tokens=256, 
        generation_config=None, 
        pad_token_id=pipe.tokenizer.eos_token_id
    )
    
    return output[0]['generated_text'][-1]['content']

## suggestion : max_new_tokens=256, generation_config=None, pad_token_id=pipe.tokenizer.eos_token_id

### Role-Based Prompting via System Instructions

Change the **system prompt** to control the model’s behavior without changing the user question.

Using the `ask_llm` function and the **system role**, ask the question:

"What is a large language model?"

Then generate answers from:
- a **PhD researcher in AI**
- a **low-literacy pirate captain**

Compare the outputs and observe how the system prompt changes the style and explanation.

In [6]:
## Your answer here
user_q = "What is a large language model?"

sys_phd = "You are a PhD resercher in AI. Please explain."
ans_phd = ask_llm(sys_phd, user_q)
print("PhD Answer: ", ans_phd)

sys_pirate = "You are a low-literacy pirate captain. Talk like a pirate."
ans_pirate = ask_llm(sys_pirate, user_q)
print("Pirate Answer: ", ans_pirate)


PhD Answer:  A large language model (LLM) is a powerful artificial intelligence system that can understand, generate, and manipulate human language. It's a type of AI that has been trained on massive amounts of text data to develop a broad understanding of language. 

Here's a breakdown of its key characteristics:

* **Understanding of Language:** LLMs excel at understanding the nuances, context, and meaning of human language. They can interpret the intent behind a sentence, identify the source of a question, and generate relevant and coherent responses. This understanding allows them to handle a wide range of tasks, from simple text generation to complex reasoning and creative writing.

* **Generalization and Adaptability:** LLMs are trained on vast datasets and can generalize their knowledge to new situations and domains. This allows them to perform well in a variety of tasks, including question answering, translation, summarization, and code generation. They can adapt their response

The system prompt really changed how the model talks. The PhD researcher answer is very formal and uses academic words like Generalization and datasets. It feels like reading a textbook or a scientific article. 

On the other hand, the pirate captain starts with pirate words like "Ahoy, matey" and "landlubbers". The tone is much more pirate like and sounds like a character in a movie. Although, I noticed that the pirate still uses some smart words like sophisticated and nuance in the middle, so maybe the model forgot it was supposed to be "low-literacy" at some point! 

This shows that the system instruction acts like a character sheet for the AI. It changes its tone and vocabulry completly even though the user question was exactly the same.

### In-Context Learning (30 points)

#### Question

##### **Q1 :**

a) What is **In-context learning** ? 

In-context learning (ICL) is when we teach the language model how to do a task just by giving it instructions and some exampels in the prompt. We don’t actually train the model or change its weights. It just learns from the context we give it in our message.

b) What are the pros and cons of ICL compared to fine-tuning? (Give 2 for each.)

Pros:

We don’t need to do any heavy training, so it saves time and computer power.

We don’t need thousands of examples like we do for fine-tunning. Just a few is enough.

Cons:

The prompt gets really long if we put too many examples, and models have a limit on how much text they can process at a given time (context window).

It might not perform as well as a fully fine-tuned model on very complex or specific tasks because it’s only using its general knowlege.

c) Write 4 techniques used in in-context learning.

Zero-shot Prompting: This is when we just give the model the instruction and the input, but we don’t show it any examples of the output. We are just relying on what it already knows from its training. Like asking it to translate a sentence right away.

Few-shot Prompting: Here we give the model a small number of examples (the “shots”) in the prompt before asking our actual question. It helps show the model the exact format or patern we want it to follow.

Chain-of-Thought (CoT): Instead of just asking for the final answer, we tell the model to show its intermediate steps. We can do this by just saying “let’s think step by step”. It helps the model think out loud so it doesn’t make simple mistakes.

Self-Consistancy: This is like an upgrade to CoT. We ask the model the same CoT prompt multiple times to get a few different reasoning paths. Then we just look at all the final answers and take a majority vote.

d) What is **Chain-of-Thought** prompting, and how does it help an AI 'think out loud' to get the right answer?

---

Use this question:<br>

"If I have 3 apples and buy 2 more, then eat 1, how many do I have?"<br>

First, run it with a simple prompt using the `ask_llm` function. <br>
Observe the answer. It will probably be wrong.  because the model does not show its reasoning.<br>

Then, use **chain-of-thought (CoT) prompting** by adding the ***short famous phrase*** at the end !<br>

Compare the two outputs. You should see that the CoT prompt leads to a more complete and correct answer .

In [9]:
## Your answer here

q = "If I have 3 apples and buy 2 more, then eat 1, how many do I have?"
sys_prompt = "You are a helpful assistant"

ans1 = ask_llm(sys_prompt, q) 
print("Simple: ", ans1)

cot_q = q + " lets think step by step."
ans2 = ask_llm(sys_prompt, cot_q)
print("CoT: ", ans2)

Simple:  You have 3 apples + 2 more = 5 apples.

CoT:  Here's how to solve this:

1. **Start with the initial amount:** 3 apples
2. **Buy 2 more:** 3 + 2 = 5 apples
3. **Eat 1:** 5 - 1 = 4 apples

You now have 4 apples.

**Answer:** You have 4.


CoT was explained above I have just copy and pasted it here:

Chain-of-Thought (CoT): Instead of just asking for the final answer, we tell the model to show its intermediate steps. We can do this by just saying “let’s think step by step”. It helps the model think out loud so it doesn’t make simple mistakes.

Comparin the outputs, the simple prompt was wrong. It just added 3 and 2 to get 5 and completly forgot about the eating part. But the CoT prompt worked perfectly. Because it wrote down the steps one by one (start with 3, add 2, subtract 1), it didn't forget the last part and got the right answer of 4.

##### When Chain-of-Thought Fails and Few-Shot Helps

Consider this question:

"A farmer has 12 sheep. All but 4 run away. How many are left?"

First, try a simple prompt 
then try a chain-of-thought (CoT) prompt.

Observe that both answers are incorrect. 


To correct this, use **few-shot prompting**, to teach the correct meaning of "all but X".

In [10]:
## Your answer here

q2 = "A farmer has 12 sheep. All but 4 run away. How many are left?"
sys_prompt = "You are a helpful assistant"

ans_simple = ask_llm(sys_prompt, q2)
print("Simple: ", ans_simple)

cot_q2 = q2 + " lets think step by step."
ans_cot = ask_llm(sys_prompt, cot_q2)
print("CoT: ", ans_cot)

Simple:  There are 12 sheep.
All but 4 run away.
So the number of sheep that are left is 12 - 4 = 8.

CoT:  Here's how to solve this:
* **Start:** 12 sheep
* **All but 4:** 12 - 4 = 8 sheep
* **Remaining:** 8 sheep

**Answer:** There are 8 sheep left.


In [14]:
better_prompt = """
Question: I have 10 cars. All but 3 are broken. How many are working?
Answer: 3. The phrase "all but 3" means exactly 3 are left.

Question: A guy has 5 apples. All but 1 are rotten. How many are good?
Answer: 1. Because "all but 1" means 1 is left.

Question: A farmer has 12 sheep. All but 4 run away. How many are left?
Answer:
"""
## Your answer here

ans_few_shot = ask_llm(sys_prompt, better_prompt)
print("Few-Shot: ", ans_few_shot)

Few-Shot:  Here's how to solve the problem:

* **Sheep:** 12 sheep - 4 sheep = 8 sheep
* **Good:** 8 sheep - 1 sheep = 7 sheep

Therefore, there are 7 good sheep.



In [ ]:
even_better_prompt2 = """
Here are some examples of how to understand the phrase "all but":

Example 1:
Question: I have 10 cars. All but 3 are broken. How many are working?
Answer: Lets think step by step. The phrase "all but 3" means every car is broken except for 3 of them. So exactly 3 cars are working. The answer is 3.

Example 2:
Question: A guy has 5 apples. All but 1 are rotten. How many are good?
Answer: Lets think step by step. "All but 1" means every apple is rotten except for 1. That means exactly 1 is good. The answer is 1.

Now answer the real question based on the examples above:

Question: A farmer has 12 sheep. All but 4 run away. How many are left?
Answer: Lets think step by step.
"""

ans_best2 = ask_llm(sys_prompt, even_better_prompt2)
print("Final Few-Shot CoT: ", ans_best2)

Final Few-Shot CoT:  Okay, I've analyzed the provided examples. Here's a breakdown of how to understand the phrase "all but":

*   **"All but":** This phrase means "except for" or "not included in." It implies that something is not available or has been excluded from the statement.

*   **"3":** This is the number of cars that are not broken.

*   **"1":** This is the number of apple that are not rotten.

*   **"12":** This is the number of sheep that are not rotten.

*   **"All but 4":** This refers to the number of sheep that are not broken.

*   **"How many are left?"** This refers to the number of remaining sheep.

The phrase "all but" is a way to express that something is not present or available.


In [19]:
sys_math = "You are a logic solver. Follow the exact pattern of the examples."

strict_prompt = """
Q: I have 10 cars. All but 3 break down. How many are left?
A: The phrase "all but 3" means exactly 3 are left. Answer: 3.

Q: A guy has 5 apples. All but 1 are rotten. How many are left?
A: The phrase "all but 1" means exactly 1 is left. Answer: 1.

Q: A farmer has 12 sheep. All but 4 run away. How many are left?
A: 
"""

ans_final = ask_llm(sys_math, strict_prompt)
print("Different sys Few-Shot: ", ans_final)

Different sys Few-Shot:  The phrase "all but 4" means exactly 4 are left. Answer: 4.



I tried giving it examples with long explanations (like explaining why cars and apples were left). The model got confused, mixed up the items, and gave us "7 good sheep." Then We tried adding "lets think step by step" to the long examples. The model overthought it and gave bad responses.I then tried separating the examples with clear labels ("Example 1", "Real Question"), but it just started hallucinating again and said stuff like "sheep that are not rotten." Finally, I changed the system prompt to "logic solver" and made the examples super short and formulaic using just "Q:" and "A:".

It seems that small models (like this 270m one) get overwhelmed easily. If you give them too much text or talk to them like a human in the prompt, they lose the main point and get distracted by the extra words. To make Few-Shot work on tiny models, the pattern has to be strict, short, and repetitive so it can just copy the format. Also, telling the model exactly what its role is in the system prompt (like a strict math solver) makes a noticable difference.

#### Question

**Q2 :**

a) According to what you see, what is the limitiation of COT ?

According to what we just tested, a big limitation of CoT is that it forces the model to look for a step-by-step mathematical process even when it's just a word trick. For the sheep question, CoT saw the numbers 12 and 4, assumed it needed to subtract them to just have a step to show, completely missing the actual meaning of the phrase "all but 4". Also, as we saw with the few-shot attempt, CoT can cause smaller models to overthink and hallucinate complex stuff instead of just giving a simple answer.

b) When does few-shot fail?

---

Now, design a prompt to solve the following problem correctly.


Target question:
"**A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost ?**"

Your goal is to guide the model to produce the correct answer by using **in-context learning (ICL)** techniques in the prompt.

In [25]:
sys_math = "You are a helpful math solver."

merged_prompt = """
--- EXAMPLE ---
Question: A pen and a pencil cost $1.50 in total. The pen costs $1.00 more than the pencil. How much does the pencil cost?
Step-by-step reasoning: Let the cost of the pencil be x. The cost of the pen is x + 1.00. The total is x + (x + 1.00) = 1.50. So 2x + 1.00 = 1.50. 2x = 0.50. x = 0.25. The answer is $0.25.

--- YOUR TASK ---
Question: A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost?
Step-by-step reasoning:
"""

ans_bat_fixed = ask_llm(sys_math, merged_prompt)
print("Bat and Ball: ", ans_bat_fixed)
## Your answer here


Bat and Ball:  Let the cost of the ball be x. The cost of the bat is x + 1.00. The total is x + (x + 1.00) = 1.10. So 2x + 1.00 = 1.10. 2x = 0.10. x = 0.05. The answer is $0.05.



---

---

#### Question

As you saw, the name of the model is **`google/gemma-3-270m-it`**.  

**Q3 :** 

a) What does the **`it`** at the end of the name refer to?  

The "it" stands for Instruction Tuned. It means the model isn't just a raw text predictor that only knows how to produce sentences. Instead, it has been specially trained to understand and follow user commands, answer questions, and act like a helpful assistant in a chat.

b) Why and how does it improve generalization ?

It improves genralization because during the instruction tuning phase, the model is trained on tons of different tasks that are formatted as instructions (like "translate this", "summarize that", "solve this math problem"). Because it learns the underlying pattern of how to follow a command, it can generalize and figure out how to do completely new tasks that it hasn't seen before, just by reading the instruction we give it.

c) Provide an example to show how it works...

If you give a base (non-tuned) model the prompt: "Translate the word apple to French:", it might just try to autocomplete it by writing something like "Translate the word banana to Spanish:" because it thinks it's a list. But an instruction-tuned model knows you are giving it an order, so it will actually follow the command and just answer the translated word.

d) Can we say this technique is a subset of in-context learning ? why ?

No, I don't think so. In-context learning (like the few-shot stuff we just did) happens purely in the prompt without changing the model's actual weights. But instruction tuning is an actual training step where the model's weights are permanently updated to learn how to follow instructions. So one is just a prompting trick, and the other is actual training.

## 2) Fine tuning using PEFT (60 points)

Training large language models normally requires updating all of their parameters, which can involve hundreds of millions of weights. This full fine‑tuning approach is slow, memory‑intensive, and often impractical for CPU‑only, offline environments. However, most pretrained models like ALBERT already capture a strong understanding of language, and only a small amount of task‑specific adjustment is needed for tasks such as sentiment classification. Parameter‑Efficient Fine‑Tuning (PEFT) provides a solution by freezing the original model and learning only a tiny set of additional weights—known as adapters—dramatically reducing training cost while preserving the integrity of the base model.

In this question, we apply PEFT using the LoRA method to fine‑tune ALBERT‑base on the IMDb Small dataset for binary sentiment classification. The model is loaded, and only the lightweight LoRA parameters are trained, making the process fast and feasible. After training, we compare the performance of the base ALBERT model with the LoRA‑enhanced version to show how a small number of learnable parameters can significantly improve task performance. This demonstrates how modern PEFT techniques enable efficient, practical fine‑tuning without the computational overhead of traditional methods.

### Dataset Loading (5 points)

The dataset used in this project is the IMDb Small movie review dataset, which is a binary sentiment classification dataset. It contains English movie reviews labeled as either positive (1) or negative (0). Each sample consists of a full text review and its corresponding sentiment label.

First, we have to load the dataset. In order to do this go to the following webpage and download the dataset. You can find it under the name "IMDB_reviews.csv".

https://github.com/LawrenceDuan/IMDb-Review-Analysis

After downloading the dataset, load it locally in your code. Display five random samples from the dataset to get a quick sense of the text and labels. Then print the total number of samples as well as the class distribution for both sentiment classes. Also check the average and maximum review length (number of tokens).

**In order to make the training process less computationally heavy, you can use only half of the dataset**

In [26]:
## Your answer here
import pandas as pd

df = pd.read_csv("IMDb-Review-Analysis-master/IMDb_Reviews.csv")
df

,review,sentiment
0,My family and I normally do not watch local mo...,1
1,"Believe it or not, this was at one time the wo...",0
2,"After some internet surfing, I found the ""Home...",0
3,One of the most unheralded great works of anim...,1
4,"It was the Sixties, and anyone with long hair ...",0
...,...,...
49995,the people who came up with this are SICK AND ...,0
49996,"The script is so so laughable... this in turn,...",0
49997,"""So there's this bride, you see, and she gets ...",0
49998,Your mind will not be satisfied by this nobud...,0


In [28]:
df_half = df.sample(frac=0.5, random_state=24)

print(df_half.sample(5))

print("Total samples:", len(df_half))

print("Classes:")
print(df_half['sentiment'].value_counts())

df_half['length'] = df_half['review'].apply(lambda x: len(str(x).split()))
print("Average length:", df_half['length'].mean())
print("Max length:", df_half['length'].max())

                                                  review  sentiment
40597  I was China in this film. I choose the screen ...          0
32761  For years I hesitated watching this movie. Now...          0
47784  Is this the movie??? Is this what Indians are ...          0
35376  Wow, "The Curse of Michael Myers" what a great...          1
44574  For those of you looking for the crazy stunts ...          1
Total samples: 25000
Classes:
sentiment
1    12587
0    12413
Name: count, dtype: int64
Average length: 230.65488
Max length: 2278


Split your dataset into train and test sets.

In [29]:
## Your answer here
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(df_half, test_size=0.2, random_state=24)

print(len(train_data))
print(len(test_data))

20000
5000


Before training ALBERT with LoRA, we must convert our raw text data into a format that the model understands. HuggingFace models expect tokenized inputs in tensor form, along with labels for supervised learning. Therefore, we need to build a custom Dataset class.

Write code to create a PyTorch Dataset and DataLoader for the IMDb dataset. Your class should tokenize the text inputs using the tokenizer, apply truncation and padding, and return the tokenized tensors along with their corresponding labels.

Then apply this dataloader class to both your training and test sets and create DataLoaders for each.

Finally, print the number of batches in the training and test DataLoaders to confirm that the data pipeline is working correctly.

In [31]:
## Your answer here

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("albert-base-v2")

class IMDbDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }

train_dataset = IMDbDataset(train_data['review'], train_data['sentiment'], tokenizer)
test_dataset = IMDbDataset(test_data['review'], test_data['sentiment'], tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print("Train batches:", len(train_loader))
print("Test batchs:", len(test_loader))

tokenizer_config.json: 0.00B [00:00, ?B/s]

C:\Users\SMani24\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SMani24\.cache\huggingface\hub\models--albert-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/760k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Train batches: 1250
Test batchs: 313


### Model Loading (5 points)

To load the model offline, we will use a locally accessible mirror that hosts the transformers version of ALBERT base v1.

Before loading the model in Python, visit https://devneeds.ir and follow the guide provided there for downloading models. For this assignment, you need to download albert/albert-base-v1. After running the required commands in your terminal and completing the download, you can load the model using the code below. Replace model_path with the path on your own system. Make sure the path points directly to the snapshots folder of the downloaded model; otherwise, the model may not load correctly.

In [34]:
from transformers import AutoModelForSequenceClassification

os.environ["HF_ENDPOINT"] = "https://hf.devneeds.ir/"

model_path = snapshot_download(repo_id="albert/albert-base-v1")

num_labels = 2

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
base_model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=num_labels, local_files_only=True)

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

tf_model.h5:   0%|          | 0.00/63.0M [00:00<?, ?B/s]

pytorch_model.bin:  66%|######6   | 31.5M/47.4M [00:00<?, ?B/s]

Some weights of AlbertForSequenceClassification were not initialized from the model checkpoint at C:\Users\SMani24\.cache\huggingface\hub\models--albert--albert-base-v1\snapshots\082438ba120d36b97b9288772a41144e941705b9 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Model Evaluation before Fine-Tuning (5 points)

Before applying PEFT and fine‑tuning the model, we first want to evaluate how the base ALBERT model performs on the dataset. Write code that runs the model on the test set and reports its accuracy, since this is a classification task.

In [36]:
## Your answer here

from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

base_model.to(device)

def evaluate_model(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            
            predictions = torch.argmax(outputs.logits, dim=1)
            
            total += labels.size(0)
            correct += (predictions == labels).sum().item()

    accuracy = correct / total
    return accuracy

base_accuracy = evaluate_model(base_model, test_loader, device)
print(f"Base model accuracy: {base_accuracy * 100}%")

Using device: cuda


Evaluating: 100%|██████████| 313/313 [00:43<00:00,  7.16it/s]

Base model accuracy: 48.4%


The base mode accuracy was 48.4%

### Adding LoRA (5 points)

Now its time to add LoRA to our model and start the fine-tuning process. LoRA is a Parameter‑Efficient Fine‑Tuning (PEFT) method. Instead of updating all the weights of a large model during training, LoRA adds a pair of small, trainable low‑rank matrices to certain layers (usually attention layers). During fine‑tuning, only these small matrices are updated while the original model weights stay frozen. This makes training much faster, uses far less memory, and allows fine‑tuning even on CPU‑only machines.

To create the LoRA matrices, use the LoraConfig function from the peft library. Below are the key parameters you need to specify:

* r : The rank of the LoRA matrices; controls how many new trainable parameters are added.

* lora_alpha : A scaling factor that adjusts the strength of the LoRA updates.

* lora_dropout : Dropout applied inside LoRA layers to reduce overfitting.

* target_modules : The specific ALBERT layers that you should set as [“query”, “key”, “value”, “dense”] where LoRA adapters are injected.

While increasing r and lora_alpha generally improves the model’s capacity and potentially its performance, higher values also make training more computationally expensive. Since this project is designed to run on a CPU and may be completed without access to a GPU, it is recommended to keep these values relatively small. Fortunately, LoRA often produces strong and stable results even with low ranks.

In [37]:
!pip install peft

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/680.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/680.7 kB ? eta -:--:--
   --------------- ------------------------ 262.1/680.7 kB ? eta -:--:--
   ------------------------------ --------- 524.3/680.7 kB 1.4 MB/s eta 0:00:01
   ---------------------------------------- 680.7/680.7 kB 1.7 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [39]:
## Your answer here

from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    task_type="SEQ_CLS",
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query", "key", "value", "dense"],
    bias="none"
)

After defining your LoraConfig, apply it to your model and then check how many parameters become trainable. Calculate both the absolute number of trainable parameters and their percentage relative to the total parameters of the model. This will show how much smaller the LoRA‑augmented model is compared to full fine‑tuning and help illustrate the efficiency benefits of PEFT.

In [40]:
## Your answer here
peft_model = get_peft_model(base_model, lora_config)

peft_model.print_trainable_parameters()

trainable params: 99,842 || all params: 11,784,964 || trainable%: 0.8472


### Fine-tuning (10 points)

In the following section you have to perform LoRA fine‑tuning of the ALBERT model on the IMDb dataset. Consider following points:
* Set the model to training mode using peft_model.train() so that layers such as dropout behave correctly during training.

* Define the AdamW optimizer with a learning rate of 2e-4 to update the trainable parameters of the model.

* Train the model for a 2 epochs.

* Track the loss during training and compute the average loss at the end of each epoch to observe how the model improves over time.

In [42]:
from torch.optim import AdamW

optimizer = AdamW(peft_model.parameters(), lr=2e-4)
epochs = 2

for epoch in range(epochs):
    peft_model.train()
    total_loss = 0
    
    for batch in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        
        outputs = peft_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        
        loss = outputs.loss
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} Average Loss: {avg_loss}")

Training Epoch 1: 100%|██████████| 1250/1250 [07:09<00:00,  2.91it/s]


Epoch 1 Average Loss: 0.3558148743033409


Training Epoch 2: 100%|██████████| 1250/1250 [07:16<00:00,  2.86it/s]

Epoch 2 Average Loss: 0.27760298127233984


### Fine-tuned Model Evaluation (5 points)

Now evaluate your fine‑tuned model and compare the results to the evaluation you performed before fine‑tuning. Check whether updating only a small number of LoRA parameters has improved the model’s performance, and analyze how much impact this lightweight fine‑tuning had on metrics such as accuracy.

In [43]:
## Your answer here
final_accuracy = evaluate_model(peft_model, test_loader, device)
print(f"Fine-tuned Model Accuracy: {final_accuracy:.4f}")

Evaluating: 100%|██████████| 313/313 [00:47<00:00,  6.57it/s]

Fine-tuned Model Accuracy: 0.8856


Before fine-tuning, the base ALBERT model had an accuracy of 48.4% on the test set, meaning it had no understanding of positive or negative sentiment and was essentially guessing. After applying LoRA and training for just 2 epochs, the accuracy jumped to 88.56%. This shows that updating only a tiny fraction of the model's weights (less than 1%) was enough to drastically improve its performance

### Compare results (10 points)

Select five random samples from the dataset and compare the predictions of your base model with those of your fine‑tuned model. For each example, analyze the two outputs, decide which model is correct, and briefly explain why. This will help you understand how fine‑tuning affected the model’s behavior on real inputs.

In [45]:
## Your answer here
import random

sample_indices = random.sample(range(len(test_dataset)), 5)
peft_model.eval()

for i, idx in enumerate(sample_indices):
    sample = test_dataset[idx]
    
    input_ids = sample['input_ids'].unsqueeze(0).to(device)
    attention_mask = sample['attention_mask'].unsqueeze(0).to(device)
    actual_label = sample['labels'].item()
    
    text = tokenizer.decode(sample['input_ids'], skip_special_tokens=True)
    
    with torch.no_grad():
        with peft_model.disable_adapter():
            base_outputs = peft_model(input_ids=input_ids, attention_mask=attention_mask)
            base_pred = base_outputs.logits.argmax(dim=-1).item()
            
        peft_outputs = peft_model(input_ids=input_ids, attention_mask=attention_mask)
        peft_pred = peft_outputs.logits.argmax(dim=-1).item()

    print("Sample number:", i+1)
    print("Text:", text)
    print("Real Label:", actual_label)
    print("Base prediction:", base_pred)
    print("Finetuned prediction:", peft_pred)
    print("*********************")

Sample number: 1
Text: in the first transformation scene, what is the music? i've heard it was "the greeks don't get no freeks". is that right? i really liked that sound. i also liked the "hyde's got nothing to hide" in the final scenes. truly a doper movie, but with many laughs and puns, sight gags, and slapstick. madam woo-woo's place was reminiscent of some places i have visited myself. ivy venus has appeared in some other stuff that is truly amazing. she looks much different in the movie we're talking about, but her maturity didn't hurt her a bit. mark blankfield reminds one of gene wilder in some ways. bess armstrong was beautiful and maintained her beauty for a long time.
Real Label: 1
Base prediction: 1
Finetuned prediction: 1
*********************
Sample number: 2
Text: i am probably a little too old for this movie, 16, and being a guy, the story is a little too girlie, but i really thought it was a cool movie overall. all the girls in this movie are so hot and really great act

<span style="color:blue;">## Your analysis here In this part we compared the base model and the finetuned model on 5 random reviews from the testset. The results show that finetuning makes the model much better. The base ALBERT model only guessed 3 out of 5 correctly, but our PEFT model got all 5 right! For example in sample 2 and 4 the base model thought the review was negative (0) but the actual label was positive (1). The finetuned model could undrestand the sentiment much better and predicted 1 for both of them. This shows that training with LoRA really helps the model learn the IMDb dataset better than just using the normal pre-trained weights.</span>

Write code that computes the following four metrics for both the base model and the fine‑tuned LoRA model on the test set: Accuracy, Precision, Recall, F1‑score


After computing the metrics, write a short analysis addressing:

* Which metric improved the most after fine‑tuning?

* Did any metric stay the same or get worse? What might explain this?

* Based on the metric differences, explain whether LoRA helped mostly with

    * reducing false positives,
    * reducing false negatives,
    * or improving confidence across both classes.

In [47]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

peft_model.to(device)

def evaluate_model_metrics(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            
            predictions = torch.argmax(outputs.logits, dim=1)
            
            all_preds.extend(predictions.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds)
    rec = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    
    return acc, prec, rec, f1

with peft_model.disable_adapter():
    base_acc, base_prec, base_rec, base_f1 = evaluate_model_metrics(peft_model, test_loader, device)
print(f"Base model - Accuracy: {base_acc}, Precision: {base_prec}, Recall: {base_rec}, F1: {base_f1}")

lora_acc, lora_prec, lora_rec, lora_f1 = evaluate_model_metrics(peft_model, test_loader, device)
print(f"LoRA model - Accuracy: {lora_acc}, Precision: {lora_prec}, Recall: {lora_rec}, F1: {lora_f1}")


Using device: cuda


Evaluating: 100%|██████████| 313/313 [00:45<00:00,  6.95it/s]


Base model - Accuracy: 0.484, Precision: 0.47674418604651164, Recall: 0.1778391167192429, F1: 0.25904652498564046


Evaluating: 100%|██████████| 313/313 [00:50<00:00,  6.21it/s]

LoRA model - Accuracy: 0.8856, Precision: 0.8899920571882446, Recall: 0.8836750788643533, F1: 0.886822318955283


<span style="color:blue;">## Your analysis here
The base model’s accuracy was 0.484, which is basicly random guessing for a binary task. The most dramatic improvement was in Recall, which jumped massively from 0.177 to 0.883! This tells us that the base model was predicting ‘0’ (negative) most of the time and missing almost all the actual positive reviews (huge amount of false negatives). The LoRA fine-tuning fixed this imbalance completely, bringing both Precision and Recall up to around 0.88. So, LoRA helped the most with reducing false negatives and balancing the model’s predictions.</span>

### Questions (15 points)

#### 1- Analyze the generalization shift caused by LoRA. Using your evaluation and the 5‑sample comparison task, write a detailed analysis covering:

* Do the LoRA adapters make the model more confident, less confident, or just more correct?

* Are the errors of the fine‑tuned model qualitatively different than those of the base?

* Does the fine‑tuned model pick up dataset‑specific biases (for example over‑fitting to certain words or patterns)?

* Based on observed predictions, what linguistic features seem to be encoded by the adapter updates?

<span style="color:blue;">## Your answer here</span>

Confidence vs Correctness: The adapters definitely made the model much more correct, as the accuracy jumped from 0.484 to 0.8856. I think it also makes it more confident, but to actually prove that, we need to check the softmax probabilities instead of just the final label. I’m assuming the probabilities for the winning class got pushed much closer to 1.0 after training.

Qualitative differences in errors: The base model’s errors were likely because it didn’t understand the task at all and just defaulted to negative predictions (since its recall was so low). The fine-tuned model’s errors are probably on tricky langauge, like heavy sarcasm or reviews that compare a bad movie to a good one, confusing the model with mixed sentiment words.

Dataset-specific biases: Yes, it almost certainly picks up some biases. For example, it might over-fit to specific words like "waste" or "masterpiece" as absolute indicators. It could even memorize actor or director names if they appear mostly in negative or positive reviews in this specific IMDb sample.

Linguistic features: Based on the huge jump in performance, the adapter updates seem to encode a strong understading of sentiment heavy adjectives (like boring, amazing, etc.). They also likely learned to capture negations better (e.g., was not good), which the base ALBERT might not naturally link to sentiment without this specific training.

#### 2- Now that you have tried fine-tuning using PEFT, when do you think its better to use full fine-tuning and when is it better to use PEFT methods such as LoRA?

<span style="color:blue;">## Your answer here</span>

Full fine-tuning is probably better when you have huge computational resources (like lots of expensive GPUs) and a massive dataset. It is also better if the new task is completely different from what the model originally learned, so you need to change all its internal knowledge.

PEFT methods like LoRA are way better when you are on a budget or only have a simple laptop GPU (like us!). It is also great when the task is somewhat similar to what the model already knows, so you just need to steer it in the right direction without changing everything. It also saves a lot of storage space since the adapter files are very small.

Gemini added this point: "Professor’s Bonus Tip: Another huge advantage of LoRA is that it helps prevent catastrophic forgetting. When you fully fine-tune a model on a narrow task, it often “forgets” its general knowledge. Because LoRA freezes the base weights, the model retains its original capabilities!"

#### 3- QLoRA (Quantized Low‑Rank Adaptation) is a parameter‑efficient fine‑tuning method that enables large language models to be adapted using limited hardware resources. In this approach, the pretrained model weights are quantized to low precision (typically 4‑bit) to significantly reduce memory usage. These compressed weights remain frozen during training, while small LoRA adapter matrices are inserted into certain layers (usually attention layers). Only these adapters are trained, allowing the model to learn task‑specific behavior without updating the full set of parameters. 

QLoRA reduces the precision of the base model weights to 4‑bit while training small LoRA adapters in higher precision. How might this design affect the trade‑off between memory efficiency and model performance? Discuss situations where this approach would likely work well and situations where the quantization might significantly harm the model’s ability to learn.

<span style="color:blue;">## Your answer here</span>

The main trade-off is getting massive memory efficency at the cost of a slight drop in accuracy. Because the base model is reduced to 4-bit, it rounds off and loses some of the fine details in its original weights.

This approach works really well for general language tasks like sentiment analysis, chat bots, or summarization, where getting the exact perfect word isn’t always required and general context is enough.

However, the 4-bit quantization might significantly harm the model if the task needs extreme precision. For example, doing complex math calculations, writing very specific code, or translating a medical document where a slight change in a word’s meaning could be dangerous. In those cases, the lost details from 4-bit compression might make it fail.

Gemini also mentioned this: "Professor’s Note: In practice, QLoRA is surprisingly resilient because it uses a specialized 4-bit data type called NF4 (NormalFloat4) which is mathematically optimized for the normal distribution of neural network weights, along with double quantization to save even more memory. But the fundamental trade-off you described remains the universal rule!"

# **Note on LLM usage:**

I used Gemini 3.1 Pro (from GapGPT) to help me with this assignment. I only used to check my answers, teach the syntax and a few conceptual questions here and there. I made a GapGPT project and set the following as the main prompt (It is probably the system prompt but I'm not sure how GapGPT works) so the LLM would only help me with learning and not just giving the answer: (This prompt was also generate by Gemini as an improved version of my own prompt :) )



The link to the conversation: https://gapgpt.app/share/8a38e9a6-08df-4b5b-8c60-e25e810e280e

[Role Description]
You are an expert AI/ML Professor and Senior Machine Learning Engineer. Your goal is to help the user complete their university LLM course assignments and learn the underlying concepts. The user already knows Python, but you must actively teach them frameworks like PyTorch, Hugging Face `transformers`, LangChain, and the OpenAI API.

[Core Objective]
By default, you are a strictly educational mentor. Do NOT provide direct, full solutions to assignment questions. Instead, guide the user to the answer by explaining concepts, providing documentation links, offering small conceptual code snippets, and asking leading questions. 

CRITICAL BEHAVIORS & PACING:
- Break complex LLM/PyTorch concepts down step-by-step. Never rush through multiple phases of an assignment at once. Provide a hint or a small piece of the puzzle, then PAUSE and ask the user to implement it or share their thoughts before moving on.
- API Accuracy: When teaching LangChain, Hugging Face, or OpenAI APIs, ensure your code uses the most modern, up-to-date syntax. Do not hallucinate deprecated API endpoints.

STATE MANAGEMENT & CONTEXT:
Assignments will often span multiple sessions. 
- At the beginning of a session, check if the user has provided an `Assignment_State.md` file or text. If so, read it to understand exactly where the last session left off.
- When the user indicates they are done for the day, generate a brief State Tracking Document formatted as markdown (e.g., `Assignment_State.md`). This document must include: 1) The overarching goal of the assignment, 2) What has been successfully implemented so far, 3) The exact bugs or next steps currently pending, and 4) Any specific libraries currently imported.

FORMATTING & STANDARDS:
- Format your responses with a Jupyter Notebook workflow in mind: use clear markdown for explanations, followed by distinct `python` code blocks for implementation.
- You MUST use $...$ or $$...$$ delimiters for ALL math expressions, formulas, and technical calculations (e.g., $$loss = -\frac{1}{N} \sum_{i=1}^{N} y_i \log(\hat{y}_i)$$).

ANTI-PROMPTS:
- Do NOT provide full code solutions.
- Do NOT hallucinate library parameters. If you are unsure of a specific PyTorch or HuggingFace argument, tell the user to check the documentation.
- Do NOT proceed to the next step of an assignment until the user confirms the current step is working .